# Face Recognition App Development

In [6]:
pip install face-recognition

  Using cached face_recognition-1.3.0-py2.py3-none-any.whl.metadata (21 kB)
  Using cached face_recognition_models-0.3.0-py2.py3-none-any.whl
  Using cached dlib-20.0.1.tar.gz (3.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Using cached face_recognition-1.3.0-py2.py3-none-any.whl (15 kB)
Failed to build dlib
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Building wheel for dlib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [78 lines of output]
      INFO:root:running bdist_wheel
      INFO:root:running build
      INFO:root:running build_ext
      Building extension for Python 3.11.5 (tags/v3.11.5:cce6ba9, Aug 24 2023, 14:38:34) [MSC v.1936 64 bit (AMD64)]
      Invoking CMake setup: 'cmake C:\Users\AhmedJaber\AppData\Local\Temp\pip-install-lpde1uah\dlib_d999660b25d74dd2a183c8247993c296\tools\python -DCMAKE_LIBRARY_OUTPUT_DIRECTORY=C:\Users\AhmedJaber\AppData\Local\Temp\pip-install-lpde1uah\dlib_d999660b25d74dd2a183c8247993c296\build\lib.win-amd64-cpython-311 -DDLIB_USE_FFMPEG=OFF -DPYTHON_EXECUTABLE=c:\Users\AhmedJaber\AppData\Local\Programs\Python\Python311\python.exe -DCMAKE_LIBRARY_OUTPUT_DIRECTORY_RELEASE=C:\Users\AhmedJaber\AppData\Local\Temp\pip-install-lpde1uah\dlib_d999660b25d74dd2a183c8247993c296\build\lib.win-amd64-cpython-311 -A x64'
      -- Building for: NMa

In [4]:
pip install opencv-python

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import json
import os
import cv2
import numpy as np
import face_recognition

ModuleNotFoundError: No module named 'face_recognition'

In [3]:
#mount the drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
"""
seed_db.py — Batch Face Enrollment from Static Photos
Scans the known_faces/ folder, extracts face embeddings from each image,
and saves all {name, embedding} entries into database.json.

Usage:
    1. Place one clear photo per person in known_faces/
       (e.g. ahmed.jpg, sara.png)
    2. Run:  python seed_db.py
"""

import json
import os
import face_recognition

DB_PATH = os.environ.get("DB_PATH", "/content/drive/MyDrive/Hackathon/database.json")
KNOWN_FACES_DIR = "/content/drive/MyDrive/Hackathon/known_faces"
# /content/drive/MyDrive/Hackathon/known_faces
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png"}


def load_database(path: str) -> dict:
    """Load the existing face database from disk, or create a fresh one."""
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {"persons": []}


def save_database(path: str, db: dict) -> None:
    """Persist the face database to disk."""
    with open(path, "w") as f:
        json.dump(db, f, indent=4)


def seed():
    """Scan known_faces/ and enroll every detected face into the database."""
    print("=" * 50)
    print("  DATABASE SEEDING FROM STATIC PHOTOS")
    print("=" * 50)
    print(f"[INFO] DB_PATH = {DB_PATH}")

    if not os.path.isdir(KNOWN_FACES_DIR):
        print(f"[ERROR] Directory '{KNOWN_FACES_DIR}/' not found. "
              "Please create it and add photos.")
        return

    # Collect image files
    image_files = [
        f for f in sorted(os.listdir(KNOWN_FACES_DIR))
        if os.path.splitext(f)[1].lower() in SUPPORTED_EXTENSIONS
    ]

    if not image_files:
        print(f"[WARNING] No image files found in '{KNOWN_FACES_DIR}/'. "
              "Add .jpg, .jpeg, or .png files and try again.")
        return

    print(f"[INFO] Found {len(image_files)} image(s) in '{KNOWN_FACES_DIR}/'.\n")

    db = load_database(DB_PATH)
    enrolled = 0
    skipped = 0

    for filename in image_files:
        name = os.path.splitext(filename)[0]
        filepath = os.path.join(KNOWN_FACES_DIR, filename)

        print(f"  Processing: {filename} → \"{name}\"")

        # Load image and extract face encodings
        image = face_recognition.load_image_file(filepath)
        encodings = face_recognition.face_encodings(image)

        if len(encodings) == 0:
            print(f"  [WARNING] No face detected in '{filename}'. Skipping.\n")
            skipped += 1
            continue

        if len(encodings) > 1:
            print(f"  [WARNING] Multiple faces in '{filename}'. "
                  "Using the first detected face.")

        embedding = encodings[0].tolist()

        db["persons"].append({
            "name": name,
            "embedding": embedding
        })

        enrolled += 1
        print(f"  [SUCCESS] '{name}' enrolled successfully!\n")

    save_database(DB_PATH, db)

    print("-" * 50)
    print(f"  Done!  Enrolled: {enrolled}  |  Skipped: {skipped}")
    print(f"  Total persons in database: {len(db['persons'])}")
    print("-" * 50)


if __name__ == "__main__":
    seed()


  DATABASE SEEDING FROM STATIC PHOTOS
[INFO] DB_PATH = /content/drive/MyDrive/Hackathon/database.json
[INFO] Found 3 image(s) in '/content/drive/MyDrive/Hackathon/known_faces/'.

  Processing: Ahmed Ashraf.jpeg → "Ahmed Ashraf"
  [SUCCESS] 'Ahmed Ashraf' enrolled successfully!

  Processing: Ahmed Jaber.JPG → "Ahmed Jaber"
  [SUCCESS] 'Ahmed Jaber' enrolled successfully!

  Processing: Ahmed Kamel.jpeg → "Ahmed Kamel"
  [SUCCESS] 'Ahmed Kamel' enrolled successfully!

--------------------------------------------------
  Done!  Enrolled: 3  |  Skipped: 0
  Total persons in database: 3
--------------------------------------------------


In [7]:
import base64
import io
import json
import os

import cv2
import numpy as np
import face_recognition
from PIL import Image

# ── Configuration ──────────────────────────────────────────────────────────
DB_PATH = os.environ.get("DB_PATH", "/kaggle/working/database.json")
TOLERANCE = 0.5
RESIZE_SCALE = 0.25  # Process frames at 25% resolution for speed

# ── Load enrolled faces at startup ─────────────────────────────────────────
print("=" * 50)
print("  FACE RECOGNITION ENGINE — JUPYTER MODE")
print("=" * 50)
print(f"[INFO] DB_PATH = {DB_PATH}")


def _load_database(path: str) -> dict:
    """Load the face database from disk, or return an empty one."""
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {"persons": []}


_db = _load_database(DB_PATH)
known_names = [p["name"] for p in _db["persons"]]
known_encodings = [np.array(p["embedding"]) for p in _db["persons"]]

print(f"[INFO] Loaded {len(known_names)} enrolled face(s).")
if not known_names:
    print("[WARNING] Database is empty. All faces will be labelled 'Unknown'.")
print("[INFO] Ready to process frames.\n")


# ── Helper: encode a BGR numpy frame → base64 JPEG ────────────────────────
def _encode_frame(frame: np.ndarray) -> str:
    """Convert a BGR numpy frame to a base64-encoded JPEG string."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_out = Image.fromarray(rgb)
    buf = io.BytesIO()
    pil_out.save(buf, format="JPEG", quality=80)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


# ── Core: process a single frame ──────────────────────────────────────────
def process_frame(b64_image: str) -> str:
    """
    Receive a base64 JPEG frame from the browser, run face recognition,
    and return the annotated frame as a base64 JPEG string.

    If no face is detected the original frame is returned unmodified.
    """
    # ── Decode base64 → numpy BGR array ──
    img_bytes = base64.b64decode(b64_image)
    pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    frame = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

    # ── Resize to 25 % for faster detection ──
    small = cv2.resize(frame, (0, 0), fx=RESIZE_SCALE, fy=RESIZE_SCALE)
    rgb_small = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)

    face_locs = face_recognition.face_locations(rgb_small)
    face_encs = face_recognition.face_encodings(rgb_small, face_locs)

    # No faces → return original frame unmodified
    if not face_locs:
        return _encode_frame(frame)

    # ── Match each face & draw annotations ──
    scale = int(1 / RESIZE_SCALE)

    for (top, right, bottom, left), enc in zip(face_locs, face_encs):
        name = "Unknown"
        color = (0, 0, 255)  # Red for unknown

        if known_encodings:
            matches = face_recognition.compare_faces(
                known_encodings, enc, tolerance=TOLERANCE
            )
            distances = face_recognition.face_distance(known_encodings, enc)
            best_idx = int(np.argmin(distances))
            if matches[best_idx]:
                name = known_names[best_idx]
                color = (0, 255, 0)  # Green for recognized

        # Scale bounding box back to full resolution
        top *= scale
        right *= scale
        bottom *= scale
        left *= scale

        # Bounding box
        cv2.rectangle(frame, (left, top), (right, bottom), color, 2)

        # Label background + text
        label_y = top - 10 if top - 10 > 10 else top + 20
        cv2.rectangle(
            frame, (left, label_y - 18), (right, label_y + 4), color, cv2.FILLED
        )
        cv2.putText(
            frame, name, (left + 4, label_y),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1,
        )

    return _encode_frame(frame)


  FACE RECOGNITION ENGINE — JUPYTER MODE
[INFO] DB_PATH = /kaggle/working/database.json
[INFO] Loaded 0 enrolled face(s).
[WARNING] Database is empty. All faces will be labelled 'Unknown'.
[INFO] Ready to process frames.



In [8]:
# ── Cell 1: Face Recognition Engine ─────────────────────────────────────
# Loads enrolled embeddings and defines process_frame(b64) → b64

import base64
import io
import json
import os

import cv2
import numpy as np
import face_recognition
from PIL import Image

# ── Configuration ──
DB_PATH = os.environ.get("DB_PATH", "/kaggle/working/database.json")
TOLERANCE = 0.5
RESIZE_SCALE = 0.25

# ── Load enrolled faces ──
print("=" * 50)
print("  FACE RECOGNITION ENGINE — JUPYTER MODE")
print("=" * 50)
print(f"[INFO] DB_PATH = {DB_PATH}")

def _load_database(path):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {"persons": []}

_db = _load_database(DB_PATH)
known_names = [p["name"] for p in _db["persons"]]
known_encodings = [np.array(p["embedding"]) for p in _db["persons"]]

print(f"[INFO] Loaded {len(known_names)} enrolled face(s).")
if not known_names:
    print("[WARNING] Database is empty. All faces will be labelled 'Unknown'.")
print("[INFO] Ready to process frames.\n")


def _encode_frame(frame):
    """BGR numpy → base64 JPEG string."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_out = Image.fromarray(rgb)
    buf = io.BytesIO()
    pil_out.save(buf, format="JPEG", quality=80)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def process_frame(b64_image):
    """
    base64 JPEG in → face recognition → annotated base64 JPEG out.
    Returns the original frame unmodified if no faces are detected.
    """
    img_bytes = base64.b64decode(b64_image)
    pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    frame = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

    small = cv2.resize(frame, (0, 0), fx=RESIZE_SCALE, fy=RESIZE_SCALE)
    rgb_small = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)

    face_locs = face_recognition.face_locations(rgb_small)
    face_encs = face_recognition.face_encodings(rgb_small, face_locs)

    if not face_locs:
        return _encode_frame(frame)

    scale = int(1 / RESIZE_SCALE)

    for (top, right, bottom, left), enc in zip(face_locs, face_encs):
        name = "Unknown"
        color = (0, 0, 255)

        if known_encodings:
            matches = face_recognition.compare_faces(
                known_encodings, enc, tolerance=TOLERANCE
            )
            distances = face_recognition.face_distance(known_encodings, enc)
            best_idx = int(np.argmin(distances))
            if matches[best_idx]:
                name = known_names[best_idx]
                color = (0, 255, 0)

        top *= scale
        right *= scale
        bottom *= scale
        left *= scale

        cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
        label_y = top - 10 if top - 10 > 10 else top + 20
        cv2.rectangle(frame, (left, label_y - 18), (right, label_y + 4), color, cv2.FILLED)
        cv2.putText(frame, name, (left + 4, label_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

    return _encode_frame(frame)

print("[INFO] process_frame() is ready.")


  FACE RECOGNITION ENGINE — JUPYTER MODE
[INFO] DB_PATH = /kaggle/working/database.json
[INFO] Loaded 0 enrolled face(s).
[WARNING] Database is empty. All faces will be labelled 'Unknown'.
[INFO] Ready to process frames.

[INFO] process_frame() is ready.


In [10]:
# ── Cell 2: Browser Camera + Live Recognition ───────────────────────────
from IPython.display import display, HTML

display(HTML("""
<div id="recognition-app" style="display:flex; flex-direction:column; align-items:center;
     gap:12px; font-family:system-ui,sans-serif; padding:20px;">

    <h3 style="margin:0;">📹 Live Face Recognition</h3>

    <!-- Live camera preview -->
    <video id="webcam" autoplay playsinline muted
           width="640" height="480"
           style="border:2px solid #555; border-radius:8px; background:#000;">
    </video>

    <!-- Controls -->
    <div style="display:flex; gap:10px;">
        <button id="start-btn"
                style="padding:10px 28px; font-size:15px; cursor:pointer;
                       background:#2ecc71; color:#fff; border:none; border-radius:8px;">
            ▶ Start Recognition
        </button>
        <button id="stop-btn"
                style="padding:10px 28px; font-size:15px; cursor:pointer;
                       background:#e74c3c; color:#fff; border:none; border-radius:8px;">
            ⏹ Stop
        </button>
    </div>

    <!-- Recognition output -->
    <h4 style="margin:0;">🔍 Recognition Output</h4>
    <img id="output-img" width="640" height="480"
         style="border:2px solid #2ecc71; border-radius:8px; background:#111;" />

    <!-- Status log -->
    <div id="status-log"
         style="width:640px; max-height:180px; overflow-y:auto;
                background:#1a1a2e; color:#eee; font-family:monospace; font-size:13px;
                padding:12px; border-radius:8px; white-space:pre-line;">
        Waiting for user action...
    </div>
</div>

<script>
(function() {
    // ── DOM refs ──
    var video      = document.getElementById('webcam');
    var outputImg  = document.getElementById('output-img');
    var startBtn   = document.getElementById('start-btn');
    var stopBtn    = document.getElementById('stop-btn');
    var statusLog  = document.getElementById('status-log');
    var canvas     = document.createElement('canvas');
    var ctx        = canvas.getContext('2d');

    var intervalId  = null;
    var processing  = false;
    var stream      = null;

    // ── Logging helper ──
    function log(msg) {
        var ts = new Date().toLocaleTimeString();
        statusLog.textContent = '[' + ts + '] ' + msg + '\\n' + statusLog.textContent;
    }

    // ── Check if a frame is non-black ──
    // Samples a 50x50 region from the center and checks avg pixel > 10
    function isFrameReal() {
        if (video.videoWidth === 0 || video.videoHeight === 0) return false;
        canvas.width  = 50;
        canvas.height = 50;
        var sx = Math.floor(video.videoWidth / 2) - 25;
        var sy = Math.floor(video.videoHeight / 2) - 25;
        ctx.drawImage(video, sx, sy, 50, 50, 0, 0, 50, 50);
        var data = ctx.getImageData(0, 0, 50, 50).data;
        var sum  = 0;
        for (var i = 0; i < data.length; i += 4) {
            sum += data[i] + data[i+1] + data[i+2];  // R + G + B
        }
        var avg = sum / (50 * 50 * 3);
        return avg > 10;
    }

    // ── Wait for video readyState >= 2 ──
    function waitForVideoReady() {
        return new Promise(function(resolve) {
            if (video.readyState >= 2) {
                resolve();
            } else {
                video.addEventListener('loadeddata', function onLoaded() {
                    video.removeEventListener('loadeddata', onLoaded);
                    resolve();
                });
            }
        });
    }

    // ── Warm-up: wait for 5 consecutive non-black frames ──
    function warmUp() {
        return new Promise(function(resolve) {
            var goodCount = 0;
            var attempts  = 0;
            var maxAttempts = 100;  // 10 seconds max at 100ms interval

            log('🔄 Warm-up: waiting for real frames (non-black)...');

            var checkInterval = setInterval(function() {
                attempts++;
                if (isFrameReal()) {
                    goodCount++;
                    log('  ✓ Non-black frame ' + goodCount + '/5 detected');
                } else {
                    goodCount = 0;  // reset — need 5 consecutive
                }

                if (goodCount >= 5) {
                    clearInterval(checkInterval);
                    log('✅ Warm-up complete! Camera is delivering real frames.');
                    resolve(true);
                } else if (attempts >= maxAttempts) {
                    clearInterval(checkInterval);
                    log('⚠️ Warm-up timed out after 10s. Frames may still be black.');
                    resolve(false);
                }
            }, 100);
        });
    }

    // ── Capture one frame and send to Python kernel ──
    function captureAndSend() {
        if (processing) return;
        if (video.videoWidth === 0) return;

        canvas.width  = video.videoWidth;
        canvas.height = video.videoHeight;
        ctx.drawImage(video, 0, 0);

        var dataUrl = canvas.toDataURL('image/jpeg', 0.8);
        var b64     = dataUrl.split(',')[1];

        processing = true;
        log('📤 Frame sent to Python...');

        var kernel = (typeof Jupyter !== 'undefined' && Jupyter.notebook)
                     ? Jupyter.notebook.kernel
                     : IPython.notebook.kernel;

        var code = '_b64_input = """' + b64 + '"""' + '\\n' + 'print(process_frame(_b64_input))';

        kernel.execute(code, {
            iopub: {
                output: function(msg) {
                    if (msg.msg_type === 'stream' && msg.content.name === 'stdout') {
                        var result = msg.content.text.trim();
                        if (result.length > 100) {
                            outputImg.src = 'data:image/jpeg;base64,' + result;
                            log('✅ Result received, displaying.');
                        }
                    }
                    processing = false;
                }
            }
        });

        // Safety timeout
        setTimeout(function() { processing = false; }, 8000);
    }

    // ── START button ──
    startBtn.addEventListener('click', async function() {
        if (intervalId) {
            log('⚠️ Already running.');
            return;
        }

        try {
            // Step 1: Request camera
            log('📷 Requesting camera permission...');
            stream = await navigator.mediaDevices.getUserMedia({
                video: { width: 640, height: 480, facingMode: 'user' }
            });

            video.srcObject = stream;
            video.play();
            log('📹 Camera stream started, warming up...');

            // Step 2: Wait for video element to have data
            await waitForVideoReady();
            log('📺 Video element ready (readyState=' + video.readyState + ')');

            // Step 3: Explicit 2-second delay for camera auto-exposure
            log('⏳ Waiting 2 seconds for camera auto-exposure...');
            await new Promise(function(r) { setTimeout(r, 2000); });

            // Step 4: Warm-up loop — wait for 5 non-black frames
            var warmedUp = await warmUp();

            if (!warmedUp) {
                log('⚠️ Camera may not be working. Trying recognition anyway...');
            }

            // Step 5: Start the recognition loop
            log('🟢 Camera ready, starting recognition loop (every 200ms)...');
            intervalId = setInterval(captureAndSend, 200);

        } catch(err) {
            log('❌ Camera error: ' + err.message);
            log('💡 Try the Upload method in the next cell instead.');
        }
    });

    // ── STOP button ──
    stopBtn.addEventListener('click', function() {
        if (intervalId) {
            clearInterval(intervalId);
            intervalId = null;
        }
        processing = false;

        // Stop camera stream
        if (stream) {
            stream.getTracks().forEach(function(t) { t.stop(); });
            stream = null;
        }
        video.srcObject = null;

        log('⏹ Stopped. Camera released.');
    });

    log('Ready. Press ▶ Start Recognition to begin.');
})();
</script>
"""))


In [24]:
# ── Cell 3: Upload Photo → Recognition (guaranteed to work) ─────────────
import ipywidgets as widgets

print("📤 Upload a photo to run face recognition on it.\n")

upload = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=True)
run_btn = widgets.Button(
    description='🔍 Run Recognition',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)
output_area = widgets.Output()

def on_run_clicked(btn):
    output_area.clear_output()
    with output_area:
        if not upload.value:
            print("⚠️  Please upload at least one photo first.")
            return
        
        for file_info in upload.value:
            name = file_info.name
            content = file_info.content
            
            print(f"Processing: {name}")
            b64_input = base64.b64encode(content).decode("utf-8")
            b64_result = process_frame(b64_input)
            
            display(HTML(
                f'<p style="font-weight:bold; font-family:sans-serif;">{name}:</p>'
                f'<img src="data:image/jpeg;base64,{b64_result}" '
                f'style="max-width:640px; border:2px solid #2ecc71; '
                f'border-radius:8px; margin-bottom:20px;" />'
            ))
            print()

run_btn.on_click(on_run_clicked)

display(widgets.VBox([
    upload,
    run_btn,
    output_area
]))


📤 Upload a photo to run face recognition on it.

